# Multilingual Health QA — Fine-tuning Pipeline
**Competition:** Zindi Multilingual Health Question Answering in Low-Resource African Languages  
**Model:** google/mt5-small + LoRA (PEFT)  
**Languages:** Akan, Amharic, Luganda, Swahili, English  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innocent-Gershon/multilingual-health-qa/blob/main/notebooks/02_Training_and_Inference.ipynb)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 datasets rouge-score sentencepiece protobuf accelerate

## Step 2 — Upload Data Files
Upload Train.csv, Val.csv, Test.csv using the cell below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload Train.csv, Val.csv, Test.csv

## Step 3 — Clone Repo and Set Up Paths

In [ ]:
!git clone https://github.com/Innocent-Gershon/multilingual-health-qa.git
%cd multilingual-health-qa
import sys
sys.path.insert(0, 'src')

## Step 4 — Configuration

In [ ]:
import os, random, numpy as np, torch

CFG = {
    'model_name': 'google/mt5-small',
    'max_input_length': 128,
    'max_target_length': 256,
    'batch_size': 8,
    'grad_accum': 4,
    'lr': 3e-4,
    'epochs': 3,
    'warmup_steps': 100,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'num_beams': 4,
    'seed': 42,
    'train_path': '../Train.csv',
    'val_path': '../Val.csv',
    'test_path': '../Test.csv',
    'output_dir': 'outputs/checkpoint',
    'submission_path': 'submissions/submission.csv',
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])
os.makedirs(CFG['output_dir'], exist_ok=True)
os.makedirs('submissions', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Step 5 — Load and Preprocess Data

In [ ]:
from preprocess import load_and_prepare

train_df = load_and_prepare(CFG['train_path'])
val_df   = load_and_prepare(CFG['val_path'])
test_df  = load_and_prepare(CFG['test_path'], is_test=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Sample prompt:', train_df['prompt'].iloc[10])
print('Sample output:', train_df['output'].iloc[10])

## Step 6 — Tokenizer and Model with LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
base_model = AutoModelForSeq2SeqLM.from_pretrained(CFG['model_name'])

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    target_modules=['q', 'v'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## Step 7 — Dataset Class

In [ ]:
import torch
from torch.utils.data import Dataset

class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_input, max_target, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row['prompt'], max_length=self.max_input,
            truncation=True, padding='max_length'
        )
        if not self.is_test:
            lab = self.tokenizer(
                row['output'], max_length=self.max_target,
                truncation=True, padding='max_length'
            )
            enc['labels'] = [
                -100 if t == self.tokenizer.pad_token_id else t
                for t in lab['input_ids']
            ]
        return {k: torch.tensor(v) for k, v in enc.items()}

train_dataset = QADataset(train_df, tokenizer, CFG['max_input_length'], CFG['max_target_length'])
val_dataset   = QADataset(val_df,   tokenizer, CFG['max_input_length'], CFG['max_target_length'])
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

## Step 8 — Training

In [ ]:
import numpy as np
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from evaluate import compute_rouge

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds   = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels          = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels  = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return compute_rouge(decoded_preds, decoded_labels)

training_args = Seq2SeqTrainingArguments(
    output_dir=CFG['output_dir'],
    num_train_epochs=CFG['epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    per_device_eval_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],
    learning_rate=CFG['lr'],
    warmup_steps=CFG['warmup_steps'],
    fp16=True,
    predict_with_generate=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rougeL_f1',
    logging_steps=50,
    seed=CFG['seed'],
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(CFG['output_dir'])
tokenizer.save_pretrained(CFG['output_dir'])
print('Training complete. Model saved.')

## Step 9 — Validation Evaluation

In [ ]:
from evaluate import compute_rouge, evaluate_by_subset
from tqdm import tqdm

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

val_preds = []
for i in tqdm(range(0, len(val_df), 16), desc='Evaluating val'):
    batch = val_df['prompt'].iloc[i:i+16].tolist()
    inputs = tokenizer(batch, return_tensors='pt', max_length=128, truncation=True, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, num_beams=CFG['num_beams'], no_repeat_ngram_size=3)
    val_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

overall = compute_rouge(val_preds, val_df['output'].tolist())
print('Overall validation ROUGE:', overall)

by_subset = evaluate_by_subset(val_preds, val_df['output'].tolist(), val_df['subset'].tolist())
import pandas as pd
print(pd.DataFrame(by_subset).T.sort_index())

## Step 10 — Generate Test Predictions and Save Submission

In [ ]:
test_preds = []
for i in tqdm(range(0, len(test_df), 16), desc='Generating test'):
    batch = test_df['prompt'].iloc[i:i+16].tolist()
    inputs = tokenizer(batch, return_tensors='pt', max_length=128, truncation=True, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, num_beams=CFG['num_beams'], no_repeat_ngram_size=3)
    test_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM':  test_preds,
})
submission.to_csv(CFG['submission_path'], index=False)
print(f'Submission saved: {len(submission)} rows')
submission.head(3)

## Step 11 — Download Submission

In [ ]:
from google.colab import files
files.download(CFG['submission_path'])